# MNO Training – Colab A100

Training notebook for the MLP-based Multiscale Neural Operator on the Warped-IFW
transient fluid dynamics challenge.

**Runtime:** GPU → A100 80 GB

**Workflow:**
1. Check GPU → Install deps → Mount Drive
2. Copy dataset to local `/content/` for fast I/O
3. Pull latest code from git
4. Configure & launch training
5. Copy best weights back to Drive + submission path

## 1 · GPU Check

In [1]:
import os

import torch

print("Working dir:", os.getcwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    raise RuntimeError("No GPU detected. In Colab set Runtime > Change runtime type > GPU.")

Working dir: /content
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.8


## 2 · Install Dependencies

In [2]:
%%capture
%pip install -q torch-cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
%pip install -q pyyaml matplotlib

## 3 · Mount Drive & Setup Paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# ---- paths ----
DRIVE_PROJECT = "/content/drive/MyDrive/iclr-2026"
LOCAL_PROJECT = "/content/iclr-2026"
DATASET_DRIVE = f"{DRIVE_PROJECT}/dataset_huggingface"
DATASET_LOCAL = f"{LOCAL_PROJECT}/dataset_huggingface"

os.makedirs(LOCAL_PROJECT, exist_ok=True)
print(f"Drive project: {DRIVE_PROJECT}")
print(f"Local project: {LOCAL_PROJECT}")

Mounted at /content/drive
Drive project: /content/drive/MyDrive/iclr-2026
Local project: /content/iclr-2026


## 4 · Download Dataset (first time only)

In [ ]:
# Uncomment the block below ONLY if dataset is not yet on Drive.
# from huggingface_hub import login, snapshot_download
# login(token="YOUR_HF_TOKEN", add_to_git_credential=True)
# snapshot_download(
#     repo_id="gram-competition/warped-ifw",
#     repo_type="dataset",
#     local_dir=f"{DRIVE_PROJECT}/dataset_huggingface/warped-ifw",
#     local_dir_use_symlinks=False,
#     resume_download=True,
#     max_workers=8,
# )
print("Dataset download cell (uncomment if needed)")

## 5 · Copy Dataset to Local SSD

In [ ]:
import shutil

if not os.path.isdir(DATASET_LOCAL):
    print("Copying dataset to local SSD for faster I/O ...")
    shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
    print("Done.")
else:
    n = len(os.listdir(f"{DATASET_LOCAL}/warped-ifw"))
    print(f"Dataset already on local SSD ({n} files).")

Copying dataset to local SSD for faster I/O ...


## 6 · Pull Latest Code

In [ ]:
%cd {DRIVE_PROJECT}
!git fetch && git pull

# Sync code (but NOT dataset or outputs)
!rsync -av \
    --exclude='dataset_huggingface' \
    --exclude='.git/' \
    --exclude='*__pycache__*/' \
    --exclude='outputs/' \
    {DRIVE_PROJECT}/ {LOCAL_PROJECT}/

%cd {LOCAL_PROJECT}
print("Code synced.")

## 7 · Training Configuration

Edit the `CONFIG` dict below to change hyperparameters.
The resolved config is automatically saved to the run output directory.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"

LOCAL_PROJECT_PATH = Path(LOCAL_PROJECT)
if str(LOCAL_PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_PATH))

# ============================================================================
# TRAINING CONFIG – edit values here
# ============================================================================
RUN_NAME = f"mno_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

CONFIG = dict(
    # ---- data ----
    dataset_dir   = str(LOCAL_PROJECT_PATH / "dataset_huggingface" / "warped-ifw"),
    num_points    = 32768,
    val_fraction  = 0.1,
    test_fraction = 0.1,
    boundary_point_fraction = 0.3,

    # ---- model (paper settings) ----
    latent_dim    = 128,
    num_modes     = 256,
    num_blocks    = 4,
    num_heads     = 8,
    k             = 16,

    # ---- training (paper settings) ----
    batch_size    = 4,
    lr            = 1e-3,
    weight_decay  = 1e-4,
    epochs        = 500,
    seed          = 100,
    loss_fn       = "rl2",
    lr_scheduler  = "auto",
    compile_model = True,

    # ---- system (A100 80GB) ----
    use_amp       = True,

    # ---- output ----
    runs_dir      = str(LOCAL_PROJECT_PATH / "outputs" / "runs"),
    run_name      = RUN_NAME,

    # ---- resume (set to checkpoint path or None) ----
    resume_from   = None,
)

# Print summary
print(f"Run: {RUN_NAME}")
print(f"Model: latent_dim={CONFIG['latent_dim']}, modes={CONFIG['num_modes']}, blocks={CONFIG['num_blocks']}, heads={CONFIG['num_heads']}, k={CONFIG['k']}")
print(f"Data:  points={CONFIG['num_points']}, bpf={CONFIG['boundary_point_fraction']}")
print(f"Train: bs={CONFIG['batch_size']}, lr={CONFIG['lr']}, epochs={CONFIG['epochs']}, loss={CONFIG['loss_fn']}")
print(f"Compile: {CONFIG['compile_model']}")
print(f"Resume from: {CONFIG['resume_from'] or 'None (train from scratch)'}")
print(f"Output: {CONFIG['runs_dir']}/{RUN_NAME}")

## 8 · Start Training

In [ ]:
import importlib

# Force reimport after git pull
for mod_name in [
    "models.mlp.model",
    "models.mlp.training",
    "models.mlp",
    "models",
    "src.training.loop",
    "src.training.trainer",
]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]
importlib.invalidate_caches()

import torch
torch.cuda.empty_cache()

from src.training.trainer import train, parse_args

config_path = str(LOCAL_PROJECT_PATH / "config" / "baseline.yaml")
checkpoint_path = str(
    Path(CONFIG["runs_dir"]) / CONFIG["run_name"] / "checkpoints" / "best.pt"
)
runs_dir = Path(CONFIG["runs_dir"])
run_name = CONFIG["run_name"]

# Build CLI-style argv from CONFIG dict
argv = [
    "--config", config_path,
    "--checkpoint-path", checkpoint_path,
    "--random-subsample",
]
for key, val in CONFIG.items():
    flag = f"--{key.replace('_', '-')}"
    if isinstance(val, bool):
        if val:
            argv.append(flag)
        else:
            argv.append(f"--no-{key.replace('_', '-')}")
    elif isinstance(val, list):
        argv.append(flag)
        argv.extend(str(v) for v in val)
    elif val is not None:
        argv.extend([flag, str(val)])

print("Effective argv:")
print(" ".join(argv))
print()

args = parse_args(argv)
train(args)

## 9 · Copy Outputs to Drive

Saves run outputs (checkpoints, config, logs) to Google Drive
so they persist after the Colab session ends.

In [ ]:
import shutil

local_run_dir = Path(CONFIG["runs_dir"]) / CONFIG["run_name"]
drive_output_dir = Path(DRIVE_PROJECT) / "outputs" / "runs" / CONFIG["run_name"]

if local_run_dir.exists():
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(local_run_dir), str(drive_output_dir), dirs_exist_ok=True)
    print(f"Run outputs copied to: {drive_output_dir}")

    # List saved files
    for p in sorted(drive_output_dir.rglob("*")):
        if p.is_file():
            size_mb = p.stat().st_size / 1e6
            print(f"  {p.relative_to(drive_output_dir)} ({size_mb:.1f} MB)")
else:
    print(f"No run directory found at {local_run_dir}")

## 10 · Copy Best Weights for Submission

Copies the best checkpoint to `models/mlp/state_dict.pt`
so the submission wrapper `MLP()` loads it automatically.

In [ ]:
best_ckpt = Path(CONFIG["runs_dir"]) / CONFIG["run_name"] / "checkpoints" / "best.pt"
submission_path = Path(DRIVE_PROJECT) / "models" / "mlp" / "state_dict.pt"

if best_ckpt.exists():
    submission_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(best_ckpt), str(submission_path))
    size_mb = submission_path.stat().st_size / 1e6
    print(f"Best weights copied to submission path: {submission_path} ({size_mb:.1f} MB)")
else:
    print(f"No best checkpoint found at {best_ckpt}")

## 11 · Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = Path(CONFIG["runs_dir"]) / CONFIG["run_name"] / "train_log.csv"
if not log_path.exists():
    print("No training log found. Run training first.")
else:
    df = pd.read_csv(log_path)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(df["epoch"], df["train_loss"], label="Train")
    axes[0].plot(df["epoch"], df["val_loss"], label="Val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_yscale("log")
    axes[0].set_title("Training & Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # LR schedule
    axes[1].plot(df["epoch"], df["lr"])
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Learning Rate")
    axes[1].set_title("LR Schedule")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Save plot
    fig_path = Path(CONFIG["runs_dir"]) / CONFIG["run_name"] / "training_curves.png"
    fig.savefig(fig_path, dpi=150)
    print(f"Saved to {fig_path}")
    plt.show()

    print(f"\nBest val loss: {df['val_loss'].min():.6f} (epoch {df['val_loss'].idxmin()})")

## 12 · Error vs. Distance from Airfoil Boundary

Scatter plot showing per-point prediction error magnitude as a function of distance to the nearest airfoil surface point. Reveals whether the model struggles near the wall (boundary layer) or in the freestream.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# --- project imports (re-import to be safe after training cell) ---
PROJECT_DIR = Path("/content/iclr-2026")
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from models import MLP as Model
from models.mlp.training import RelativeL2Loss
from src.data import WarpedIFWDataset, split_train_val_test, unscale_velocity_batch
from src.training import NUM_T_IN, NUM_T_OUT

# ---------- settings (inherit from the training cell when available) ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_args = globals().get("args", None)
_seed = int(getattr(_args, "seed", 42))
_num_points = int(getattr(_args, "num_points", 4096))
_val_fraction = float(getattr(_args, "val_fraction", 0.1))
_test_fraction = float(getattr(_args, "test_fraction", 0.1))
_checkpoint = globals().get("checkpoint_path", None)

DATASET_DIR = PROJECT_DIR / "dataset_huggingface" / "warped-ifw"
files = sorted(DATASET_DIR.glob("*.npz"))
if not files:
    raise FileNotFoundError(f"No .npz files in {DATASET_DIR}")

_, val_files, _ = split_train_val_test(
    files, val_fraction=_val_fraction, test_fraction=_test_fraction, seed=_seed
)

val_dataset = WarpedIFWDataset(
    val_files,
    num_points=_num_points,
    random_crop=False,
    seed=_seed,
    scaler_eps=1e-6,
)

# ---------- load trained model ----------
if _checkpoint is None or not Path(_checkpoint).exists():
    raise FileNotFoundError(
        "No checkpoint found. Run the training cell first so that "
        "`checkpoint_path` is defined and the file exists."
    )

model = Model(
    num_t_in=NUM_T_IN,
    num_t_out=NUM_T_OUT,
    latent_dim=int(getattr(_args, "latent_dim", 128)),
    num_modes=int(getattr(_args, "num_modes", 256)),
    num_heads=int(getattr(_args, "num_heads", 4)),
    num_blocks=int(getattr(_args, "num_blocks", 4)),
    k=int(getattr(_args, "k", 12)),
    knn_query_chunk_size=int(getattr(_args, "knn_query_chunk_size", 2048)),
    graph_query_chunk_size=int(getattr(_args, "graph_query_chunk_size", 2048)),
    use_torch_cluster_knn=bool(getattr(_args, "use_torch_cluster_knn", False)),
).to(device)

state_dict = torch.load(str(_checkpoint), map_location=device)
# Strip "_orig_mod." prefix added by torch.compile when saving compiled models.
state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
print(f"Loaded checkpoint: {_checkpoint}")

# ---------- collect per-point errors & distances ----------
MAX_SAMPLES = min(len(val_dataset), 20)  # cap for speed

all_distances = []
all_errors = []

with torch.no_grad():
    for i in range(MAX_SAMPLES):
        t, pos, idcs_airfoil, velocity_in, velocity_out, velocity_mean, velocity_std, wall_distance, surface_frame = val_dataset[i]

        # Move to device and add batch dim
        t = t.unsqueeze(0).to(device)
        pos_dev = pos.unsqueeze(0).to(device)
        idcs_airfoil_list = [idcs_airfoil.to(device)]
        velocity_in_dev = velocity_in.unsqueeze(0).to(device)
        velocity_out_dev = velocity_out.unsqueeze(0).to(device)
        v_mean = velocity_mean.unsqueeze(0).to(device)
        v_std = velocity_std.unsqueeze(0).to(device)

        pred_scaled = model(t, pos_dev, idcs_airfoil_list, velocity_in_dev, v_mean, v_std)

        # Un-scale to physical velocity
        pred_phys = unscale_velocity_batch(pred_scaled.float(), v_mean, v_std)
        target_phys = unscale_velocity_batch(velocity_out_dev.float(), v_mean, v_std)

        # Per-point error: L2 across velocity components, averaged over output time steps
        # pred_phys shape: (1, T_out, N, 2)
        point_error = (pred_phys - target_phys).norm(dim=-1).mean(dim=1).squeeze(0).cpu()  # (N,)

        # Use precomputed wall_distance from dataset
        all_distances.append(wall_distance.numpy())
        all_errors.append(point_error.numpy())

distances = np.concatenate(all_distances)
errors = np.concatenate(all_errors)

# ---------- scatter plot ----------
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(distances, errors, s=1, alpha=0.15, c=errors, cmap="inferno", edgecolors="none")
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Point Error (L2, physical)")
ax.set_xlabel("Distance to Nearest Airfoil Surface Point")
ax.set_ylabel("Prediction Error (L2, physical)")
ax.set_title(f"Error vs. Distance from Boundary  ({MAX_SAMPLES} val samples)")

# Optional: binned mean curve
NUM_BINS = 50
bin_edges = np.linspace(distances.min(), distances.max(), NUM_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_means = np.full(NUM_BINS, np.nan)
for b in range(NUM_BINS):
    mask = (distances >= bin_edges[b]) & (distances < bin_edges[b + 1])
    if mask.sum() > 0:
        bin_means[b] = errors[mask].mean()
valid = ~np.isnan(bin_means)
ax.plot(bin_centers[valid], bin_means[valid], color="cyan", linewidth=2, label="Binned mean")
ax.legend()

plt.tight_layout()

# Save next to the run outputs when available
_runs_dir = globals().get("runs_dir", None)
_run_name = globals().get("run_name", None)
if _runs_dir is not None and _run_name is not None:
    save_dir = Path(_runs_dir) / _run_name
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / "error_vs_distance_from_boundary.png", dpi=150)
    print(f"Saved plot to {save_dir / 'error_vs_distance_from_boundary.png'}")

plt.show()
print(f"Total points plotted: {len(distances):,}")
print(f"Mean error near boundary (d < {bin_edges[1]:.4f}): {bin_means[0]:.6f}")
print(f"Mean error far from boundary (d > {bin_edges[-2]:.4f}): {bin_means[-1]:.6f}")

## 13 · Full-Resolution Side-View Cut

Runs one full-resolution sample through the checkpoint, extracts a center plane cut, interpolates `|u|` onto a regular grid, and plots the whole predicted horizon as ground truth, prediction, and signed speed delta. The defaults produce an `x-z` side view at the center `y` plane; override `full_rollout_slice_axis`, `full_rollout_plane_x_dim`, or `full_rollout_plane_y_dim` before running the cell if you want a different cut.

In [ ]:
from contextlib import nullcontext
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.path import Path as MplPath
from scipy.interpolate import griddata
from scipy.spatial import ConvexHull, QhullError, cKDTree

PROJECT_DIR = Path("/content/iclr-2026")
if not PROJECT_DIR.exists():
    cwd = Path.cwd().resolve()
    if (cwd / "dataset_huggingface").exists():
        PROJECT_DIR = cwd
    elif (cwd.parent / "dataset_huggingface").exists():
        PROJECT_DIR = cwd.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from models import MLP as Model
from src.data import WarpedIFWDataset, split_train_val_test, unscale_velocity_batch
from src.training import NUM_POS, NUM_T_IN, NUM_T_OUT

FULL_ROLLOUT_SAMPLE_INDEX = int(globals().get("full_rollout_sample_index", 0))
FULL_ROLLOUT_SPLIT = str(globals().get("full_rollout_split", "val")).lower()
FULL_ROLLOUT_CHECKPOINT = globals().get("full_rollout_checkpoint", None)
FULL_ROLLOUT_PLANE_X_DIM = int(globals().get("full_rollout_plane_x_dim", 0))
FULL_ROLLOUT_PLANE_Y_DIM = int(globals().get("full_rollout_plane_y_dim", 2))
full_rollout_remaining_dims = [
    dim for dim in range(3) if dim not in {FULL_ROLLOUT_PLANE_X_DIM, FULL_ROLLOUT_PLANE_Y_DIM}
]
FULL_ROLLOUT_SLICE_AXIS = int(
    globals().get(
        "full_rollout_slice_axis",
        full_rollout_remaining_dims[0] if len(full_rollout_remaining_dims) == 1 else 1,
    )
)
FULL_ROLLOUT_GRID_NX = int(globals().get("full_rollout_grid_nx", 360))
FULL_ROLLOUT_GRID_NY = int(globals().get("full_rollout_grid_ny", 220))
FULL_ROLLOUT_PLANE_MIN_POINTS = int(globals().get("full_rollout_plane_min_points", 1800))
FULL_ROLLOUT_PLANE_START_FRAC = float(globals().get("full_rollout_plane_start_frac", 0.0025))
FULL_ROLLOUT_PLANE_MAX_FRAC = float(globals().get("full_rollout_plane_max_frac", 0.08))
FULL_ROLLOUT_AIRFOIL_PLANE_FACTOR = float(globals().get("full_rollout_airfoil_plane_factor", 2.0))

if len({FULL_ROLLOUT_SLICE_AXIS, FULL_ROLLOUT_PLANE_X_DIM, FULL_ROLLOUT_PLANE_Y_DIM}) != 3:
    raise ValueError("Slice axis and in-plane dimensions must be three distinct axes from {0, 1, 2}.")

COORD_LABELS = ("x", "y", "z")

FLOW_CMAP = "jet"
DELTA_CMAP = "RdBu_r"
FLOW_PANEL_BG = "#f6f1e8"
FIG_BG = "#efe7da"
TEXT_COLOR = "#231a15"
MUTED_TEXT = "#6f6258"


def adaptive_plane_mask(axis_values, axis_center, min_points=1200, start_frac=0.0025, max_frac=0.08):
    axis_span = float(axis_values.max() - axis_values.min())
    frac = start_frac
    while True:
        tol = max(1e-10, axis_span * frac)
        mask = np.abs(axis_values - axis_center) <= tol
        if int(mask.sum()) >= min_points or frac >= max_frac:
            return mask, tol, frac
        frac *= 1.8


def build_hull_polygon(points_2d):
    if points_2d.shape[0] < 3:
        return None
    try:
        hull = ConvexHull(points_2d)
    except QhullError:
        return None
    return points_2d[hull.vertices]


def split_airfoil_components(points_2d):
    if points_2d.shape[0] < 3:
        return []

    min_component_size = max(12, points_2d.shape[0] // 80)
    if points_2d.shape[0] <= min_component_size:
        return [points_2d]

    tree = cKDTree(points_2d)
    neighbor_count = min(8, points_2d.shape[0])
    distances, _ = tree.query(points_2d, k=neighbor_count)
    distances = np.atleast_2d(distances)
    local_scale = float(np.nanmedian(distances[:, -1]))
    span = float(np.linalg.norm(points_2d.max(axis=0) - points_2d.min(axis=0)))
    if not np.isfinite(local_scale) or local_scale <= 0.0:
        local_scale = max(span * 0.02, 1e-6)
    connection_radius = max(local_scale * 1.6, span * 1e-3, 1e-8)

    remaining = np.ones(points_2d.shape[0], dtype=bool)
    components = []

    for start_idx in np.where(remaining)[0]:
        if not remaining[start_idx]:
            continue
        stack = [int(start_idx)]
        remaining[start_idx] = False
        component_indices = []

        while stack:
            idx = stack.pop()
            component_indices.append(idx)
            neighbor_indices = tree.query_ball_point(points_2d[idx], r=connection_radius)
            for neighbor_idx in neighbor_indices:
                if remaining[neighbor_idx]:
                    remaining[neighbor_idx] = False
                    stack.append(int(neighbor_idx))

        if len(component_indices) >= min_component_size:
            component = points_2d[np.asarray(component_indices, dtype=np.int64)]
            components.append(component)

    if not components:
        components = [points_2d]

    components.sort(key=lambda component: float(np.median(component[:, 0])))
    return components


def prepare_slice_geometry(points_3d, airfoil_points_3d):
    slice_values = points_3d[:, FULL_ROLLOUT_SLICE_AXIS]
    slice_center = float(np.median(slice_values))
    plane_mask, plane_tol, plane_frac = adaptive_plane_mask(
        slice_values,
        slice_center,
        min_points=FULL_ROLLOUT_PLANE_MIN_POINTS,
        start_frac=FULL_ROLLOUT_PLANE_START_FRAC,
        max_frac=FULL_ROLLOUT_PLANE_MAX_FRAC,
    )

    cut_points = points_3d[plane_mask]
    plane_points = cut_points[:, [FULL_ROLLOUT_PLANE_X_DIM, FULL_ROLLOUT_PLANE_Y_DIM]]
    if plane_points.shape[0] < 50:
        raise RuntimeError(
            f"Too few points on the selected plane cut ({plane_points.shape[0]}). "
            "Increase the tolerance or choose another sample."
        )

    x_min, x_max = map(float, (plane_points[:, 0].min(), plane_points[:, 0].max()))
    y_min, y_max = map(float, (plane_points[:, 1].min(), plane_points[:, 1].max()))
    grid_x = np.linspace(x_min, x_max, FULL_ROLLOUT_GRID_NX)
    grid_y = np.linspace(y_min, y_max, FULL_ROLLOUT_GRID_NY)
    X, Y = np.meshgrid(grid_x, grid_y)

    inside_domain = np.ones(X.shape, dtype=bool)
    domain_poly = build_hull_polygon(plane_points)
    if domain_poly is not None:
        domain_path = MplPath(domain_poly, closed=True)
        inside_domain = domain_path.contains_points(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape)

    airfoil_plane_mask = (
        np.abs(airfoil_points_3d[:, FULL_ROLLOUT_SLICE_AXIS] - slice_center)
        <= FULL_ROLLOUT_AIRFOIL_PLANE_FACTOR * plane_tol
    )
    airfoil_plane_points = airfoil_points_3d[airfoil_plane_mask][:, [FULL_ROLLOUT_PLANE_X_DIM, FULL_ROLLOUT_PLANE_Y_DIM]]

    airfoil_polygons = []
    inside_airfoil = np.zeros(X.shape, dtype=bool)
    for component in split_airfoil_components(airfoil_plane_points):
        poly = build_hull_polygon(component)
        if poly is None:
            continue
        airfoil_polygons.append(poly)
        poly_path = MplPath(poly, closed=True)
        inside_airfoil |= poly_path.contains_points(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape)

    return {
        "plane_mask": plane_mask,
        "plane_points": plane_points,
        "X": X,
        "Y": Y,
        "inside_domain": inside_domain,
        "inside_airfoil": inside_airfoil,
        "airfoil_polygons": airfoil_polygons,
        "slice_center": slice_center,
        "plane_tol": plane_tol,
        "plane_frac": plane_frac,
        "cut_point_count": int(plane_mask.sum()),
    }


def interpolate_scalar_to_slice(slice_geometry, scalar_values):
    cut_values = scalar_values[slice_geometry["plane_mask"]]
    X = slice_geometry["X"]
    Y = slice_geometry["Y"]
    plane_points = slice_geometry["plane_points"]

    linear = griddata(plane_points, cut_values, (X, Y), method="linear")
    nearest = griddata(plane_points, cut_values, (X, Y), method="nearest")
    filled = np.where(np.isnan(linear), nearest, linear)

    mask = (~slice_geometry["inside_domain"]) | slice_geometry["inside_airfoil"]
    return np.ma.array(filled, mask=mask)


def concatenate_masked_values(masked_arrays):
    compressed = [values.compressed() for values in masked_arrays if values.count()]
    if not compressed:
        return np.array([0.0], dtype=np.float64)
    return np.concatenate(compressed)


rollout_args = globals().get("args", None)
rollout_checkpoint_value = FULL_ROLLOUT_CHECKPOINT or globals().get("checkpoint_path", None)
if rollout_checkpoint_value is None:
    raise FileNotFoundError(
        "No checkpoint path is available. Run the training setup cell first or set `full_rollout_checkpoint`."
    )

rollout_checkpoint_path = Path(rollout_checkpoint_value)
if not rollout_checkpoint_path.exists():
    alternative_checkpoint_path = PROJECT_DIR / rollout_checkpoint_path
    if alternative_checkpoint_path.exists():
        rollout_checkpoint_path = alternative_checkpoint_path
    else:
        raise FileNotFoundError(f"Checkpoint not found: {rollout_checkpoint_path.resolve()}")

rollout_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rollout_use_amp = bool(getattr(rollout_args, "use_amp", True)) and rollout_device.type == "cuda"
rollout_seed = int(getattr(rollout_args, "seed", 42))
rollout_val_fraction = float(getattr(rollout_args, "val_fraction", 0.1))
rollout_test_fraction = float(getattr(rollout_args, "test_fraction", 0.1))
rollout_scaler_eps = float(getattr(rollout_args, "scaler_eps", 1e-6))

rollout_dataset_dir = PROJECT_DIR / "dataset_huggingface" / "warped-ifw"
if not rollout_dataset_dir.exists():
    raise FileNotFoundError(f"Dataset directory not found: {rollout_dataset_dir.resolve()}")

rollout_files = sorted(rollout_dataset_dir.glob("*.npz"))
if not rollout_files:
    raise FileNotFoundError(f"No .npz files found in {rollout_dataset_dir}")

rollout_train_files, rollout_val_files, rollout_test_files = split_train_val_test(
    rollout_files,
    val_fraction=rollout_val_fraction,
    test_fraction=rollout_test_fraction,
    seed=rollout_seed,
)
rollout_split_to_files = {
    "train": rollout_train_files,
    "val": rollout_val_files,
    "test": rollout_test_files,
}
if FULL_ROLLOUT_SPLIT not in rollout_split_to_files:
    raise ValueError(
        f"Unsupported split '{FULL_ROLLOUT_SPLIT}'. Expected one of {sorted(rollout_split_to_files)}."
    )

rollout_selected_files = rollout_split_to_files[FULL_ROLLOUT_SPLIT]
if not rollout_selected_files:
    raise RuntimeError(f"Split '{FULL_ROLLOUT_SPLIT}' is empty.")

full_rollout_dataset = WarpedIFWDataset(
    rollout_selected_files,
    num_points=NUM_POS,
    random_crop=False,
    seed=rollout_seed,
    scaler_eps=rollout_scaler_eps,
)

FULL_ROLLOUT_SAMPLE_INDEX = max(0, min(FULL_ROLLOUT_SAMPLE_INDEX, len(full_rollout_dataset) - 1))
rollout_sample_path = rollout_selected_files[FULL_ROLLOUT_SAMPLE_INDEX]

rollout_model = Model(
    num_t_in=NUM_T_IN,
    num_t_out=NUM_T_OUT,
    latent_dim=int(getattr(rollout_args, "latent_dim", 128)),
    num_modes=int(getattr(rollout_args, "num_modes", 256)),
    num_heads=int(getattr(rollout_args, "num_heads", 4)),
    num_blocks=int(getattr(rollout_args, "num_blocks", 4)),
    k=int(getattr(rollout_args, "k", 12)),
    knn_query_chunk_size=int(getattr(rollout_args, "knn_query_chunk_size", 2048)),
    graph_query_chunk_size=int(getattr(rollout_args, "graph_query_chunk_size", 2048)),
    use_torch_cluster_knn=bool(getattr(rollout_args, "use_torch_cluster_knn", False)),
).to(rollout_device)

rollout_state_dict = torch.load(str(rollout_checkpoint_path), map_location=rollout_device)
# Strip "_orig_mod." prefix added by torch.compile when saving compiled models.
rollout_state_dict = {k.removeprefix("_orig_mod."): v for k, v in rollout_state_dict.items()}
rollout_model.load_state_dict(rollout_state_dict)
rollout_model.eval()

(
    rollout_t,
    rollout_pos,
    rollout_idcs_airfoil,
    rollout_velocity_in,
    rollout_velocity_out,
    rollout_velocity_mean,
    rollout_velocity_std,
    _rollout_wall_distance,
    _rollout_surface_frame,
) = full_rollout_dataset[FULL_ROLLOUT_SAMPLE_INDEX]

rollout_t = rollout_t.unsqueeze(0).to(rollout_device)
rollout_pos_device = rollout_pos.unsqueeze(0).to(rollout_device)
rollout_idcs_airfoil_list = [rollout_idcs_airfoil.to(rollout_device)]
rollout_velocity_in_device = rollout_velocity_in.unsqueeze(0).to(rollout_device)
rollout_velocity_out_device = rollout_velocity_out.unsqueeze(0).to(rollout_device)
rollout_velocity_mean_device = rollout_velocity_mean.unsqueeze(0).to(rollout_device)
rollout_velocity_std_device = rollout_velocity_std.unsqueeze(0).to(rollout_device)

rollout_autocast = (
    torch.autocast(device_type="cuda", dtype=torch.float16)
    if rollout_use_amp
    else nullcontext()
)

with torch.inference_mode():
    with rollout_autocast:
        rollout_pred_scaled = rollout_model(
            rollout_t,
            rollout_pos_device,
            rollout_idcs_airfoil_list,
            rollout_velocity_in_device,
            rollout_velocity_mean_device,
            rollout_velocity_std_device,
        )

rollout_pred_phys = unscale_velocity_batch(
    rollout_pred_scaled.float(),
    rollout_velocity_mean_device,
    rollout_velocity_std_device,
).squeeze(0).cpu()
rollout_target_phys = unscale_velocity_batch(
    rollout_velocity_out_device.float(),
    rollout_velocity_mean_device,
    rollout_velocity_std_device,
).squeeze(0).cpu()

rollout_error_phys = rollout_pred_phys - rollout_target_phys
rollout_target_mag = rollout_target_phys.norm(dim=-1).numpy()
rollout_pred_mag = rollout_pred_phys.norm(dim=-1).numpy()
rollout_delta_mag = rollout_pred_mag - rollout_target_mag

rollout_pos_np = rollout_pos.cpu().numpy()
rollout_airfoil_np = rollout_pos[rollout_idcs_airfoil.long()].cpu().numpy()
rollout_slice = prepare_slice_geometry(rollout_pos_np, rollout_airfoil_np)

rollout_target_slices = []
rollout_pred_slices = []
rollout_delta_slices = []
rollout_per_t_rl2 = []
rollout_per_t_delta_abs = []

for ts in range(NUM_T_OUT):
    rollout_target_t = rollout_target_phys[ts]
    rollout_error_t = rollout_error_phys[ts]
    rollout_denom = rollout_target_t.norm(dim=-1).sum().clamp(min=1e-8)
    rollout_per_t_rl2.append((rollout_error_t.norm(dim=-1).sum() / rollout_denom).item())

    target_slice = interpolate_scalar_to_slice(rollout_slice, rollout_target_mag[ts])
    pred_slice = interpolate_scalar_to_slice(rollout_slice, rollout_pred_mag[ts])
    delta_slice = interpolate_scalar_to_slice(rollout_slice, rollout_delta_mag[ts])

    rollout_target_slices.append(target_slice)
    rollout_pred_slices.append(pred_slice)
    rollout_delta_slices.append(delta_slice)

    delta_values = delta_slice.compressed()
    rollout_per_t_delta_abs.append(float(np.mean(np.abs(delta_values))) if delta_values.size else 0.0)

rollout_mean_rl2 = float(np.mean(rollout_per_t_rl2))
rollout_mean_delta_abs = float(np.mean(rollout_per_t_delta_abs))

rollout_flow_values = concatenate_masked_values(rollout_target_slices + rollout_pred_slices)
rollout_delta_values = concatenate_masked_values(rollout_delta_slices)

rollout_flow_vmax = max(float(np.percentile(rollout_flow_values, 99.5)), 1e-8)
rollout_delta_vmax = max(float(np.percentile(np.abs(rollout_delta_values), 99.5)), 1e-8)

rollout_flow_norm = Normalize(vmin=0.0, vmax=rollout_flow_vmax)
rollout_delta_norm = TwoSlopeNorm(vcenter=0.0, vmin=-rollout_delta_vmax, vmax=rollout_delta_vmax)

plt.style.use("seaborn-v0_8-white")
fig, axes = plt.subplots(
    NUM_T_OUT,
    3,
    figsize=(18, 3.55 * NUM_T_OUT),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)
fig.patch.set_facecolor(FIG_BG)

for ax in axes.flat:
    ax.set_facecolor(FLOW_PANEL_BG)
    ax.set_aspect("equal")
    ax.tick_params(axis="both", colors=MUTED_TEXT, labelsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)

plane_x_label = COORD_LABELS[FULL_ROLLOUT_PLANE_X_DIM]
plane_y_label = COORD_LABELS[FULL_ROLLOUT_PLANE_Y_DIM]
slice_axis_label = COORD_LABELS[FULL_ROLLOUT_SLICE_AXIS]

panel_titles = [
    "Ground Truth |u|",
    "Prediction |u|",
    "Delta |u| (pred - truth)",
]

for ts in range(NUM_T_OUT):
    gt_ax, pred_ax, delta_ax = axes[ts]

    gt_ax.pcolormesh(
        rollout_slice["X"],
        rollout_slice["Y"],
        rollout_target_slices[ts],
        shading="auto",
        cmap=FLOW_CMAP,
        norm=rollout_flow_norm,
    )
    pred_ax.pcolormesh(
        rollout_slice["X"],
        rollout_slice["Y"],
        rollout_pred_slices[ts],
        shading="auto",
        cmap=FLOW_CMAP,
        norm=rollout_flow_norm,
    )
    delta_ax.pcolormesh(
        rollout_slice["X"],
        rollout_slice["Y"],
        rollout_delta_slices[ts],
        shading="auto",
        cmap=DELTA_CMAP,
        norm=rollout_delta_norm,
    )

    for ax in (gt_ax, pred_ax, delta_ax):
        for poly in rollout_slice["airfoil_polygons"]:
            ax.fill(
                poly[:, 0],
                poly[:, 1],
                facecolor="#111111",
                edgecolor="#f7f3eb",
                linewidth=0.9,
                zorder=5,
            )
        ax.set_xlim(float(rollout_slice["X"].min()), float(rollout_slice["X"].max()))
        ax.set_ylim(float(rollout_slice["Y"].min()), float(rollout_slice["Y"].max()))

    gt_ax.text(
        0.02,
        0.98,
        f"t+{ts + 1}\nRL2={rollout_per_t_rl2[ts]:.4f}\nmean |delta|={rollout_per_t_delta_abs[ts]:.4f}",
        transform=gt_ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        color=TEXT_COLOR,
        fontweight="bold",
        bbox=dict(
            facecolor=(1.0, 0.98, 0.95, 0.86),
            edgecolor="none",
            boxstyle="round,pad=0.35",
        ),
        zorder=6,
    )

    if ts == 0:
        for ax, title in zip((gt_ax, pred_ax, delta_ax), panel_titles):
            ax.set_title(title, fontsize=13, color=TEXT_COLOR, fontweight="bold", pad=12)

    gt_ax.set_ylabel(plane_y_label, fontsize=11, color=TEXT_COLOR)
    if ts == NUM_T_OUT - 1:
        gt_ax.set_xlabel(plane_x_label, fontsize=11, color=TEXT_COLOR)
        pred_ax.set_xlabel(plane_x_label, fontsize=11, color=TEXT_COLOR)
        delta_ax.set_xlabel(plane_x_label, fontsize=11, color=TEXT_COLOR)

rollout_flow_sm = ScalarMappable(norm=rollout_flow_norm, cmap=FLOW_CMAP)
rollout_flow_sm.set_array([])
rollout_delta_sm = ScalarMappable(norm=rollout_delta_norm, cmap=DELTA_CMAP)
rollout_delta_sm.set_array([])

rollout_flow_cbar = fig.colorbar(
    rollout_flow_sm,
    ax=axes[:, :2].ravel().tolist(),
    fraction=0.022,
    pad=0.018,
    shrink=0.98,
)
rollout_flow_cbar.set_label("Velocity magnitude |u|", color=TEXT_COLOR)
rollout_flow_cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(rollout_flow_cbar.ax.get_yticklabels(), color=TEXT_COLOR)

rollout_delta_cbar = fig.colorbar(
    rollout_delta_sm,
    ax=axes[:, 2].ravel().tolist(),
    fraction=0.028,
    pad=0.018,
    shrink=0.98,
)
rollout_delta_cbar.set_label("Signed delta |u|", color=TEXT_COLOR)
rollout_delta_cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(rollout_delta_cbar.ax.get_yticklabels(), color=TEXT_COLOR)

rollout_model_label = (
    rollout_checkpoint_path.parents[1].name
    if len(rollout_checkpoint_path.parents) >= 2
    else rollout_checkpoint_path.stem
)
fig.suptitle(
    f"Full-Resolution Side-View Cut | {FULL_ROLLOUT_SPLIT} sample {FULL_ROLLOUT_SAMPLE_INDEX} | {rollout_sample_path.stem}",
    fontsize=18,
    fontweight="bold",
    color=TEXT_COLOR,
)

fig.text(
    0.5,
    0.992,
    (
        f"Checkpoint: {rollout_model_label} | "
        f"{plane_x_label}-{plane_y_label} cut at {slice_axis_label}={rollout_slice['slice_center']:.5f} +/- {rollout_slice['plane_tol']:.2e} | "
        f"mean rollout RL2={rollout_mean_rl2:.4f} | mean |delta|={rollout_mean_delta_abs:.4f}"
    ),
    ha="center",
    va="top",
    fontsize=11,
    color=MUTED_TEXT,
)

rollout_default_save_dir = (
    rollout_checkpoint_path.parents[1]
    if len(rollout_checkpoint_path.parents) >= 2
    else rollout_checkpoint_path.parent
)
rollout_save_dir = Path(globals().get("full_rollout_save_dir", rollout_default_save_dir))
rollout_save_dir.mkdir(parents=True, exist_ok=True)
rollout_save_path = rollout_save_dir / (
    f"full_rollout_side_cut_{FULL_ROLLOUT_SPLIT}_sample_{FULL_ROLLOUT_SAMPLE_INDEX:03d}.png"
)

fig.savefig(
    "/content/drive/MyDrive/full_rollout_visualization2.png",
    dpi=1080,
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)
plt.show()
print(f"Loaded checkpoint: {rollout_checkpoint_path}")
print(f"Visualized file: {rollout_sample_path.name}")
print(
    f"Slice: {plane_x_label}-{plane_y_label} plane at {slice_axis_label}={rollout_slice['slice_center']:.6f} "
    f"+/- {rollout_slice['plane_tol']:.6e} ({100.0 * rollout_slice['plane_frac']:.3f}% of axis span)"
)
print(f"Cut points used: {rollout_slice['cut_point_count']}")
print(f"Airfoil sections detected: {len(rollout_slice['airfoil_polygons'])}")
print(f"Saved figure to: {rollout_save_path}")
print(f"Per-timestep RL2: {[round(value, 4) for value in rollout_per_t_rl2]}")
print(f"Per-timestep mean |delta|: {[round(value, 4) for value in rollout_per_t_delta_abs]}")

## 14 · Test Evaluation (Full Resolution)

Evaluate the best checkpoint on a held-out test set at full resolution.
Reports per-sample and per-timestep Relative L2 error.

In [ ]:
import sys
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch

PROJECT_DIR = Path("/content/iclr-2026")
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from models import MLP as Model
from src.data import WarpedIFWDataset, split_train_val_test, unscale_velocity_batch
from src.training import NUM_T_IN, NUM_T_OUT

# ---------- settings (inherit from the training cell when available) ----------
_args = globals().get("args", None)
_seed = int(getattr(_args, "seed", 42))
_val_fraction = float(getattr(_args, "val_fraction", 0.1))
_test_fraction = float(getattr(_args, "test_fraction", 0.1))
_checkpoint = globals().get("checkpoint_path", None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = bool(getattr(_args, "use_amp", True)) and device.type == "cuda"

DATASET_DIR = PROJECT_DIR / "dataset_huggingface" / "warped-ifw"
files = sorted(DATASET_DIR.glob("*.npz"))
if not files:
    raise FileNotFoundError(f"No .npz files in {DATASET_DIR}")

_, _, test_files = split_train_val_test(
    files, val_fraction=_val_fraction, test_fraction=_test_fraction, seed=_seed
)
print(f"Test set: {len(test_files)} samples")

from src.training import NUM_POS
test_dataset = WarpedIFWDataset(
    test_files,
    num_points=NUM_POS,
    random_crop=False,
    seed=_seed,
    scaler_eps=float(getattr(_args, "scaler_eps", 1e-6)),
)

# ---------- load model ----------
if _checkpoint is None or not Path(_checkpoint).exists():
    raise FileNotFoundError("No checkpoint found. Run the training cell first.")

model = Model(
    num_t_in=NUM_T_IN,
    num_t_out=NUM_T_OUT,
    latent_dim=int(getattr(_args, "latent_dim", 128)),
    num_modes=int(getattr(_args, "num_modes", 256)),
    num_heads=int(getattr(_args, "num_heads", 4)),
    num_blocks=int(getattr(_args, "num_blocks", 4)),
    k=int(getattr(_args, "k", 12)),
    knn_query_chunk_size=int(getattr(_args, "knn_query_chunk_size", 2048)),
    graph_query_chunk_size=int(getattr(_args, "graph_query_chunk_size", 2048)),
    use_torch_cluster_knn=bool(getattr(_args, "use_torch_cluster_knn", False)),
).to(device)

state_dict = torch.load(str(_checkpoint), map_location=device)
state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
print(f"Loaded checkpoint: {_checkpoint}")

# ---------- predict on full test set ----------
autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

per_sample_rl2 = []
per_sample_per_t_rl2 = []

with torch.inference_mode():
    for i in range(len(test_dataset)):
        t, pos, idcs_airfoil, vel_in, vel_out, vel_mean, vel_std, _wd, _sf = test_dataset[i]

        t_dev = t.unsqueeze(0).to(device)
        pos_dev = pos.unsqueeze(0).to(device)
        idcs_list = [idcs_airfoil.to(device)]
        vel_in_dev = vel_in.unsqueeze(0).to(device)
        vel_out_dev = vel_out.unsqueeze(0).to(device)
        v_mean_dev = vel_mean.unsqueeze(0).to(device)
        v_std_dev = vel_std.unsqueeze(0).to(device)

        with autocast_ctx:
            pred_scaled = model(t_dev, pos_dev, idcs_list, vel_in_dev, v_mean_dev, v_std_dev)

        pred_phys = unscale_velocity_batch(pred_scaled.float(), v_mean_dev, v_std_dev).squeeze(0).cpu()
        target_phys = unscale_velocity_batch(vel_out_dev.float(), v_mean_dev, v_std_dev).squeeze(0).cpu()

        # Per-timestep RL2: (T_out,)
        err = pred_phys - target_phys
        reduce_dims = tuple(range(1, pred_phys.ndim))
        t_num = torch.linalg.vector_norm(err, dim=reduce_dims)
        t_den = torch.linalg.vector_norm(target_phys, dim=reduce_dims).clamp(min=1e-8)
        t_rl2 = (t_num / t_den).numpy()
        per_sample_per_t_rl2.append(t_rl2)

        # Overall RL2 for this sample
        overall_num = torch.linalg.vector_norm(err.reshape(-1))
        overall_den = torch.linalg.vector_norm(target_phys.reshape(-1)).clamp(min=1e-8)
        per_sample_rl2.append(float((overall_num / overall_den).item()))

        if (i + 1) % 10 == 0 or i == len(test_dataset) - 1:
            print(f"  [{i+1}/{len(test_dataset)}] sample RL2={per_sample_rl2[-1]:.6f}")

per_sample_rl2 = np.array(per_sample_rl2)
per_sample_per_t_rl2 = np.stack(per_sample_per_t_rl2, axis=0)  # (N_test, T_out)

print(f"\n{'='*60}")
print(f"Test set RL2 ({len(test_dataset)} samples, full resolution)")
print(f"{'='*60}")
print(f"Mean RL2 (across samples):       {per_sample_rl2.mean():.6f}")
print(f"Median RL2:                       {np.median(per_sample_rl2):.6f}")
print(f"Std RL2:                          {per_sample_rl2.std():.6f}")
print(f"Min / Max RL2:                    {per_sample_rl2.min():.6f} / {per_sample_rl2.max():.6f}")
print(f"Mean per-timestep RL2:            {[round(v, 6) for v in per_sample_per_t_rl2.mean(axis=0).tolist()]}")
print(f"{'='*60}")